# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [8]:
# imports
from openai import OpenAI
from dotenv import load_dotenv
import os
from IPython.display import display, Markdown, update_display
from scraper import fetch_website_contents, fetch_website_links

In [9]:
# constants

MODEL_GEMINI = 'gemini-3.1-flash-lite'
MODEL_LLAMA = 'llama3.2:1b'

In [10]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    print("GEMINI_API_KEY not found in environment variables. Please set it in your .env file.")
elif not api_key.startswith("AIza"):
    print("GEMINI_API_KEY does not appear to be valid. Please check your .env file.")
else:
    print("GEMINI_API_KEY is valid.")

GEMINI_API_KEY is valid.


In [11]:
system_prompt = "You are a coding assistant who will answer questions based on the provided context."
user_prompt = "Please answer the following question based on the context provided:"

In [28]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
def fetch_website_contents(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

"""
user_prompt += "\n\n" + question

In [34]:
def QnA(provider):
   gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai", api_key=api_key)
   ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollam")

   if provider == "gemini":
      client = gemini
      model = MODEL_GEMINI
   elif provider == "ollama":
      client = ollama
      model = MODEL_LLAMA
   else:
      raise ValueError("provider must be 'gemini' or 'ollama'")

   stream = client.chat.completions.create(
             model=model,
             messages=[
                       {"role": "system", "content": system_prompt},
                       {"role": "user", "content": user_prompt}
                      ],
             stream=True
            )

   result = ""
   display_handle = display(Markdown(result), display_id=True)
   for chunk in stream:
      result += chunk.choices[0].delta.content or ''
      update_display(Markdown(result), display_id=display_handle.display_id)


In [32]:
# Get gemini-3.1-flash-lite to answer, with streaming
QnA(provider="gemini")

This function, `fetch_website_contents(url)`, is designed to scrape and clean the text content from a webpage. It is likely being used to prepare web data for a Large Language Model (LLM) or a similar natural language processing task.

Here is a breakdown of what the code does and why:

### 1. Fetching the Data
*   **`requests.get(url, headers=headers)`**: It sends an HTTP GET request to the provided URL. The `headers` (assumed to be defined globally) are likely included to mimic a real web browser, preventing the server from blocking the request (e.g., using a `User-Agent`).
*   **`BeautifulSoup(response.content, "html.parser")`**: It parses the raw HTML content into a tree structure that can be navigated and manipulated easily using the BeautifulSoup library.

### 2. Extracting the Title
*   **`title = soup.title.string if soup.title else "No title found"`**: It safely checks if a `<title>` tag exists. If it does, it extracts the text; otherwise, it assigns a default string.

### 3. Cleaning the Content
*   **Removing Noise (`irrelevant.decompose()`)**:
    *   The code iterates through `script`, `style`, `img`, and `input` tags within the `<body>`.
    *   **Why:** These tags contain code, metadata, or media that does not constitute human-readable text. Removing them reduces "noise" and ensures the resulting data focuses only on the actual written content.
*   **`soup.body.get_text(separator="\n", strip=True)`**:
    *   It extracts all remaining text from the body, using a newline as a separator to maintain structure and removing leading/trailing whitespace.
    *   **Why:** This converts the messy HTML into a clean, plain-text format that is easier for models to read and process.

### 4. Formatting and Truncation
*   **`return (title + "\n\n" + text)[:2_000]`**:
    *   It combines the title and the body text with a clear separator.
    *   **Truncation (`[:2_000]`)**: It limits the output to the first 2,000 characters.
    *   **Why:** AI models and databases have "context window" limits. Truncating the data ensures that the input remains within a manageable size and prevents the process from consuming excessive memory or tokens.

In [35]:
# Get Llama 3.2 to answer through the local Ollama server
QnA(provider="ollama")

The provided code is designed to extract the title and text content of a webpage. Here's a breakdown of each function and their corresponding reasons:

**Function 1: `fetch_website_contents(url)`**
```python
def fetch_website_contents(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="","strip=True")
    else:
        text = ""
    return (title + "

" + text)[:2_000]
```
This function:

1. Sends a GET request to the specified `url` with the provided `headers` (in this case, an empty dictionary `headers=None`).
2. Creates a BeautifulSoup object from the received HTML content using the `html.parser`.
3. Extracts the webpage title by retrieving the `<title>` tag from the soup.
4. If the webpage has a non-empty `<body>` section, it cleans up repetitive HTML elements (`script`, `style`, `img`, and `input`) by removing them.
5. Retrieves the clean text content from the `<body>` section using the `get_text()` method with the specified `separator` (`","` and `strip=True`). The sliced text is then appended to the title string.
6. If the webpage has no non-empty `<body>` section, it returns the original title string with 2,000 characters truncated from the end.

**Function 2: Does not exist in original code**

The second function is almost identical to the first one. The differences are:

* The function does not handle cases where the webpage has an empty `<body>` section.
* It does not return a message with the title and text content.
* The last two lines of the function are identical, with the same `text` variable name.

**Function 3: Does not exist in original code**

The third function is almost identical to the first two functions. The differences are:

* The function does not handle cases where the webpage has an empty `<body>` section.
* It does not return a message with the title and text content.
* The last two lines of the function are identical, with the same `text` variable name.

In summary, all three functions aim to extract the title and text content of a webpage. However, Function 3 is not necessary, as the third function is identical to the first two. Additionally, Function 2 does not handle cases where the webpage has an empty `<body>` section, which can lead to incorrect results.